# Tâche 2 - Classification automatique de descriptions d'incidents

Cette tâche consiste à classifier de courts textes décrivant des incidents sur des chantiers de construction. Pour chaque incident, on retrouve une étiquette qui décrit un aspect de l’incident. Nous avons substitué les étiquettes de classes par des nombres entiers. Ces étiquettes sont inconnues et vous devrez tenter de les identifier à la section 3 de ce *notebook*. 

Pour la première partie, vous devez compléter la fonction (*train_and_test_classifier*) qui entraîne un modèle (les options étant la régression logistique et le naïf bayésien) et en faire l'évaluation sur des données d'entraînement et des données de test. Pour la deuxième partie, vous devez tentez de déterminer à quoi correspond chacune des classes d’incident. 

## Section 1 - Lecture des fichiers de données

La fonction *load_incident_data* est utilisée pour lire les 2 fichiers (*train* et *test*). Rien à modifier dans cette section. 

In [1]:
import json
import pandas as pd

train_json_fn = "./data/incidents_train.json"
test_json_fn = "./data/incidents_test.json"


def load_incident_dataset(filename):
    with open(filename, 'r') as fp:
        incident_list = json.load(fp)
    return incident_list

In [3]:
train_list = load_incident_dataset(train_json_fn)
print("Nombre d'incidents:", len(train_list))
print("\nUn exemple:\n", train_list[10])
print("\nClés du premier incident:\n", train_list[0].keys())

df_train = pd.DataFrame(train_list)

print("\nNombre de documents par classe:")
print(df_train["label"].value_counts().sort_index())


Nombre d'incidents: 2475

Un exemple:
 {'text': " At approximately 12:15 p.m. on February 8  2011  Employee # 1 and four other  coworkers were working in the West storage Facility at Enterprise Products  located at Mont Belvieu  Texas. A major fire began but Employee # 1 wasn't  able to escape. The recovery effort was delayed until the evening of February  10  2011  due to the presence of hydrocarbon vapors and small residual fires.   ", 'label': '6'}

Clés du premier incident:
 dict_keys(['text', 'label'])

Nombre de documents par classe:
label
0    457
1     15
2    277
3     65
4     21
5    958
6    109
7    308
8    265
Name: count, dtype: int64


### Déséquilibre des classes
On remarque un déséquilibre important dans l’échantillon d’apprentissage. Les classes 0 (18,5 %) et surtout 5 (38,7 %) sont beaucoup plus représentées que les autres classes. 

À l’inverse, certaines classes, comme les classes 1 (0,6 %) et 4 (0,8 %), contiennent très peu d’observations. Ce déséquilibre pourrait influencer l’apprentissage des classificateurs et leur capacité à reconnaître les classes minoritaires.

In [4]:
test_list = load_incident_dataset(test_json_fn)
df_test = pd.DataFrame(test_list)

print("Nombre d'incidents", len(test_list))
incident = test_list[10]
print("\nUne description d'incident:", incident["text"])
print("\nSon étiquette:", incident["label"])

print("\nNombre de documents par classe:")
print(df_test["label"].value_counts().sort_index())

Nombre d'incidents 531

Une description d'incident:  On May 24  2004  Employee #1 was welding the deck caps on the of fourth floor  decking. The employee fell from the edge and sustained life threatening head  and internal injuries. Employee #1 was then transferred to the Kaweah Delta  District Hospital  where he subsequently died from his injuries. The employee  was not wearing any fall protection but there was a single line of wire rope  at the perimeter of the structure.                                              

Son étiquette: 5

Nombre de documents par classe:
label
0    103
1      7
2     58
3     15
4      2
5    191
6     29
7     66
8     60
Name: count, dtype: int64


## Section 2 - Entraînement et évaluation des modèles  

Vous pouvez ajouter tout le code dont vous avez besoin pour l'entraînement. Merci de ne pas modifier la signature de la fonction d'entraînement. Vous pouvez ajouter des cellules au *notebook* si nécessaire. 

In [11]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_score, recall_score, accuracy_score, confusion_matrix
from sklearn.model_selection import cross_val_score
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer


In [ ]:
def train_and_test_classifier(train_fn, test_fn, model='NB'):
    """
    :param train_fn et test_fn: les 2 fichiers utilisées pour entraîner et tester les classificateurs.
    :param model: le type de classificateur. NB = Naive Bayes, LR = Régression logistique.
    :return: un dictionnaire contenant 3 valeurs: l'accuracy à l'entraînement, l'accuracy sur le jeu de test, 
             et la matrice de confusion calculée sur les données de test.
             Vous pouvez ajouter d'autres éléments à results, par ex. les modèles entraînés.
    """
    def load_dataset(filename):
        incidents = load_incident_dataset(filename)

        texts = [incident["text"] for incident in incidents]
        labels = [incident["label"] for incident in incidents]

        return texts, labels

        
    X_train, y_train = load_dataset(train_fn)
    X_test, y_test = load_dataset(test_fn)

    #Sélection du classificateur
    if model == 'NB':
        classifier = MultinomialNB()
        
        #Vectorisation des données retenu pour le classificateur Naive Bayes
        vectorizer = CountVectorizer(lowercase=True)
        vectorizer.fit(X_train)

        #Création du vecteur de caractéristiques pour les données d'entrainement et de test
        X_train_vectorized = vectorizer.transform(X_train)
        X_test_vectorized = vectorizer.transform(X_test)

    elif model == 'LR':
        classifier = LogisticRegression(max_iter=500, class_weight='balanced')

        #Vectorisation des données retenu pour le classificateur Régression logistique
        vectorizer = TfidfVectorizer(lowercase=True)  
        vectorizer.fit(X_train)

        #Création du vecteur de caractéristiques pour les données d'entrainement et de test
        X_train_vectorized = vectorizer.transform(X_train)
        X_test_vectorized = vectorizer.transform(X_test)
    

    classifier.fit(X_train_vectorized, y_train)

    y_pred_train = classifier.predict(X_train_vectorized)
    y_pred = classifier.predict(X_test_vectorized)

    results = dict()

    results['accuracy_train'] = accuracy_score(y_train, y_pred_train)
    results['accuracy_test'] = accuracy_score(y_test, y_pred)
    results['confusion_matrix'] = confusion_matrix(y_test, y_pred)
    results['precision'] = precision_score(y_test, y_pred, average=None, zero_division=0)
    results['recall'] = recall_score(y_test, y_pred, average=None, zero_division=0)
    results['classifier'] = classifier
    results['vectorizer'] = vectorizer
    results['features'] = vectorizer.get_feature_names_out()

    return results

In [79]:
results_nb = train_and_test_classifier(
    train_json_fn,
    test_json_fn,
    model='NB'
)

print("Naive Bayes")
print("Accuracy entraînement :", results_nb['accuracy_train'])
print("Accuracy test :", results_nb['accuracy_test'])
print("Précision : ", results_nb['precision'])
print("Rappel : ", results_nb['recall'])
print("Matrice de confusion :")
print(results_nb['confusion_matrix'])

Naive Bayes
Accuracy entraînement : 0.8569696969696969
Accuracy test : 0.6760828625235404
Précision :  [0.55555556 0.         0.82089552 1.         0.         0.72653061
 0.91666667 0.5        0.625     ]
Rappel :  [0.63106796 0.         0.94827586 0.13333333 0.         0.93193717
 0.37931034 0.42424242 0.33333333]
Matrice de confusion :
[[ 65   0   0   0   0  23   0   9   6]
 [  4   0   0   0   0   1   0   2   0]
 [  0   0  55   0   0   3   0   0   0]
 [  5   0   0   2   0   3   0   4   1]
 [  0   0   0   0   0   1   0   1   0]
 [  9   0   2   0   0 178   0   0   2]
 [  1   0   4   0   0   5  11   8   0]
 [ 17   0   5   0   0  13   0  28   3]
 [ 16   0   1   0   0  18   1   4  20]]


#### Autre résultat

Naive Bayes - Vectorisation : TfidfVectorizer()

- Accuracy entraînement : 0.48
- Accuracy test : 0.4105461393596987

Matrice de confusion :
```text
[[ 20   0   0   0   0  83   0   0   0]
 [  0   0   0   0   0   7   0   0   0]
 [  0   0   8   0   0  50   0   0   0]
 [  0   0   0   0   0  15   0   0   0]
 [  0   0   0   0   0   2   0   0   0]
 [  1   0   0   0   0 190   0   0   0]
 [  0   0   0   0   0  29   0   0   0]
 [  1   0   0   0   0  65   0   0   0]
 [  1   0   0   0   0  59   0   0   0]]

In [83]:
results_lr = train_and_test_classifier(
    train_json_fn,
    test_json_fn,
    model='LR'
)

print("Logistic Regression(class_weight='balanced') - Vectorisation: TfidfVectorizer")
print("Accuracy entraînement :", results_lr['accuracy_train'])
print("Accuracy test :", results_lr['accuracy_test'])
print("Précision : ", results_lr['precision'])
print("Rappel : ", results_lr['recall'])
print("Matrice de confusion :")
print(results_lr['confusion_matrix'])



Logistic Regression(class_weight='balanced') - Vectorisation: TfidfVectorizer
Accuracy entraînement : 0.8856565656565657
Accuracy test : 0.7269303201506592
Précision :  [0.57142857 1.         0.8358209  0.625      0.33333333 0.8972973
 0.95454545 0.47368421 0.56896552]
Rappel :  [0.66019417 0.57142857 0.96551724 0.66666667 0.5        0.86910995
 0.72413793 0.40909091 0.55      ]
Matrice de confusion :
[[ 68   0   0   1   0   6   1  16  11]
 [  2   4   0   0   0   0   0   1   0]
 [  0   0  56   0   0   1   0   1   0]
 [  4   0   0  10   0   0   0   0   1]
 [  0   0   0   0   1   0   0   1   0]
 [ 16   0   2   0   0 166   0   1   6]
 [  0   0   2   2   0   3  21   1   0]
 [ 17   0   6   2   2   5   0  27   7]
 [ 12   0   1   1   0   4   0   9  33]]


#### Autres résultats

Logistic Regression - Vectorisation : TfidfVectorizer()

- Accuracy entraînement : 0.8573737373737373
- Accuracy test : 0.6779661016949152

Matrice de confusion :
```text
[[ 69   0   0   0   0  21   0  10   3]
 [  3   0   0   0   0   2   0   2   0]
 [  0   0  53   0   0   4   0   1   0]
 [  4   0   0   1   0   7   0   2   1]
 [  1   0   0   0   0   0   0   1   0]
 [ 13   0   1   0   0 177   0   0   0]
 [  3   0   4   0   0   6  13   3   0]
 [ 23   0   4   0   0  11   0  26   2]
 [ 13   0   0   0   0  20   0   6  21]]

Logistic Regression - Vectorisation : CountVectorizer()

- Accuracy entraînement : 1.0
- Accuracy test : 0.6930320150659134

Matrice de confusion :
```text
[[ 63   0   0   0   0  15   0  13  12]
 [  1   2   1   0   0   0   0   3   0]
 [  0   0  53   0   0   3   1   1   0]
 [  5   0   0   6   0   1   0   3   0]
 [  1   0   0   0   0   0   0   1   0]
 [  9   0   1   0   0 171   0   4   6]
 [  0   0   4   2   0   4  15   4   0]
 [ 17   0   3   1   1   5   1  31   7]
 [ 12   0   0   0   0  12   0   9  27]]

LogisticRegression(class_weight='balanced') - Vectorisation : CountVectorizer() 

- Accuracy entraînement : 0.9995959595959596
- Accuracy test : 0.6836158192090396

Matrice de confusion :
```text
[[ 60   0   0   0   0  14   0  16  13]
 [  1   3   0   0   0   0   1   2   0]
 [  0   0  53   0   0   3   1   1   0]
 [  5   0   0   8   0   0   0   2   0]
 [  0   0   0   0   1   0   0   1   0]
 [ 12   0   1   0   1 165   0   3   9]
 [  0   0   4   3   0   3  16   3   0]
 [ 16   0   4   1   2   4   1  30   8]
 [ 13   0   0   0   0   8   0  12  27]]

In [90]:
vectorizer_lr = results_lr['vectorizer']
classifier_lr = results_lr['classifier']

vectorizer_nb = results_nb['vectorizer']
classifier_nb = results_nb['classifier']

In [91]:
features = results_nb['features']

mots = [feature for feature in features if feature.isalpha()]

print("Nombre total d'attributs :", len(features))
print("Nombre d'attributs composés uniquement de lettres :", len(mots))
print("Exemples de mots :", mots[:20])

Nombre total d'attributs : 9838
Nombre d'attributs composés uniquement de lettres : 9071
Exemples de mots : ['aa', 'aaa', 'aaron', 'abacus', 'abandoned', 'abandonment', 'abatement', 'abdomen', 'abdominal', 'abhe', 'abi', 'abiqua', 'abl', 'ablative', 'able', 'about', 'above', 'aboveground', 'abraham', 'abrasion']


In [92]:
class_probs = list(zip(classifier_nb.classes_, classifier_nb.class_log_prior_))
for x, prob in class_probs:
    print("logprob({}) = {}".format(x, round(prob,2)))


logprob(0) = -1.69
logprob(1) = -5.11
logprob(2) = -2.19
logprob(3) = -3.64
logprob(4) = -4.77
logprob(5) = -0.95
logprob(6) = -3.12
logprob(7) = -2.08
logprob(8) = -2.23


In [120]:
df_nb = pd.DataFrame(vectorizer_nb.get_feature_names_out(), columns =['Mots']) 
for i in range(len(classifier_nb.classes_)):
    df_nb[classifier_nb.classes_[i]] = list(classifier_nb.feature_log_prob_[i])

display(df_nb[9080:9090])

,Mots,0,1,2,3,4,5,6,7,8
9080,trellis,-10.979343,-9.306559,-10.549884,-9.755567,-9.382022,-10.970601,-10.004463,-10.774655,-8.241823
9081,tremble,-10.979343,-9.306559,-10.549884,-9.755567,-9.382022,-10.970601,-10.004463,-10.774655,-10.639718
9082,trembling,-10.979343,-9.306559,-9.856737,-9.755567,-9.382022,-11.663748,-10.004463,-10.774655,-10.639718
9083,tremie,-10.979343,-9.306559,-10.549884,-9.755567,-9.382022,-10.277453,-10.004463,-10.774655,-10.639718
9084,tremmie,-10.979343,-9.306559,-10.549884,-9.755567,-9.382022,-11.663748,-10.004463,-10.774655,-9.946571
9085,trench,-5.627485,-9.306559,-8.470442,-8.146130,-8.688875,-8.830534,-8.395026,-7.555780,-7.548676
9086,trenched,-10.286196,-9.306559,-10.549884,-9.755567,-9.382022,-11.663748,-10.004463,-10.774655,-10.639718
9087,trencher,-10.286196,-9.306559,-10.549884,-9.755567,-9.382022,-11.663748,-10.004463,-9.676043,-10.639718
9088,trenches,-9.369905,-9.306559,-10.549884,-9.062420,-9.382022,-11.663748,-9.311316,-10.081508,-10.639718
9089,trenching,-9.880731,-9.306559,-10.549884,-9.755567,-9.382022,-10.970601,-9.311316,-10.774655,-9.946571


In [94]:
features_lr = results_lr['features']

mots = [feature for feature in features_lr if feature.isalpha()]

print("Nombre total d'attributs :", len(features_lr))
print("Nombre d'attributs composés uniquement de lettres :", len(mots))
print("Exemples de mots :", mots[:20])

Nombre total d'attributs : 9838
Nombre d'attributs composés uniquement de lettres : 9071
Exemples de mots : ['aa', 'aaa', 'aaron', 'abacus', 'abandoned', 'abandonment', 'abatement', 'abdomen', 'abdominal', 'abhe', 'abi', 'abiqua', 'abl', 'ablative', 'able', 'about', 'above', 'aboveground', 'abraham', 'abrasion']


In [119]:
df_lr = pd.DataFrame(
    vectorizer_lr.get_feature_names_out(),
    columns=['Mots']
)

for i in range(len(classifier_lr.classes_)):
    df_lr[classifier_lr.classes_[i]] = classifier_lr.coef_[i]

display(df_lr[9080:9090])

,Mots,0,1,2,3,4,5,6,7,8
9080,trellis,-0.111571,-0.019669,-0.046552,-0.050592,-0.022143,-0.072925,-0.053096,-0.073869,0.450418
9081,tremble,-0.001208,-0.000194,-0.001080,-0.000541,-0.000187,0.008048,-0.000526,-0.001096,-0.003215
9082,trembling,-0.004125,-0.001087,0.036004,-0.013116,-0.002509,-0.003692,-0.003779,-0.004185,-0.003511
9083,tremie,-0.009217,-0.003080,-0.005086,-0.003639,-0.002483,0.055788,-0.003170,-0.011144,-0.017968
9084,tremmie,-0.040817,-0.005070,-0.014197,-0.008488,-0.005948,-0.031259,-0.006224,-0.034034,0.146038
9085,trench,2.805810,-0.424100,-0.447005,-0.240630,-0.181099,-0.783689,-0.202948,-0.172095,-0.354244
9086,trenched,0.035955,-0.002667,-0.003852,-0.005187,-0.001092,-0.008909,-0.002699,-0.005068,-0.006482
9087,trencher,0.130697,-0.014826,-0.026384,-0.030047,-0.013815,-0.039167,-0.027992,0.112604,-0.091071
9088,trenches,0.051728,-0.025681,-0.054999,0.065718,-0.027036,-0.049943,0.154305,-0.029122,-0.084971
9089,trenching,-0.038070,-0.017641,-0.042871,-0.050114,-0.017473,0.013399,0.162934,-0.067152,0.056988


## Section 3 - À quoi correspondent les classes? Explicabilité du modèle

Vous pouvez ajouter tout le code et toutes les cellules dont vous avez besoin pour cette sous-tâche.

In [97]:
classe_0_nb = df_nb[['Mots', '0']].sort_values(
    by='0',
    ascending=False
).head(20)

print("Naive Bayes")
display(classe_0_nb)

Naive Bayes


,Mots,0
8824,the,-2.487878
9574,was,-3.443512
910,and,-3.540371
3215,employee,-3.606596
8944,to,-3.749504
6022,of,-3.821607
6049,on,-4.080628
4580,in,-4.418312
4267,he,-4.635462
1084,at,-4.830875


In [98]:
classe_0 = df[['Mots', '0']].sort_values(
    by='0',
    ascending=False
).head(20)

print("Régression logistique")
display(classe_0)

Régression logistique


,Mots,0
9085,trench,2.805810
9556,wall,1.869945
2119,collapsed,1.811790
2523,crushed,1.786331
9435,vehicle,1.749255
5270,loader,1.651271
4986,killed,1.505523
5368,machine,1.376956
9118,truck,1.373882
4062,grader,1.303656


Avec Naive Bayes, il est plus difficile de retrouver le contexte associé à cette classe avec seulement 20 mots. Les mots les plus probables sont principalement des mots très fréquents et ayant peu de contexte comme the, was, end... Le modèle de régression logistique semble plus pertinent pour interpréter les classes. Les mots ayant un coefficient élevé semble représentatif du contexte, comparativement aux mots ayant les log-probabilités les plus élevées dans le modèle Naive Bayes. Nous utiliserons que le modèle de régression logistique pour interpréter les contextes des autres classes. 

La classe 0 semble correspondre à des accidents qui impliquent un véhicule ou de la machinerie, dans lequel l'individu a été écrasé ou coincé. Le mot tranchée (trench) est celui qui possède le poids le plus élevé pour la classe 0 dans le modèle de régression logistique retenu. 


In [121]:
classe_1_nb = df_nb[['Mots', '1']].sort_values(
    by='1',
    ascending=False
).head(20)

print("Naive Bayes")
display(classe_1_nb)

Naive Bayes


,Mots,1
8824,the,-4.711439
3215,employee,-5.592987
9574,was,-5.592987
910,and,-5.723040
8944,to,-5.810052
6049,on,-5.939263
6022,of,-6.087683
4580,in,-6.262037
9588,water,-6.473346
4267,he,-6.473346


In [99]:
classe_1 = df[['Mots', '1']].sort_values(
    by='1',
    ascending=False
).head(20)

display(classe_1)

,Mots,1
3051,drowned,4.458505
1232,barge,4.072059
9588,water,2.759777
1429,boat,2.273759
7374,river,2.005949
6573,pool,1.983495
9360,upside,1.747969
4136,guardrail,1.633062
4780,into,1.622437
1433,bobcat,1.513515


La classe 1 semble correspondre à des noyades ou des accidents qui implique l'eau, soit en bateau ou à la nage. 

In [122]:
classe_2_nb = df_nb[['Mots', '2']].sort_values(
    by='2',
    ascending=False
).head(20)

print("Naive Bayes")
display(classe_2_nb)

Naive Bayes


,Mots,2
8824,the,-2.683161
9574,was,-3.527016
910,and,-3.824850
3215,employee,-3.851616
8944,to,-3.857800
6022,of,-4.133151
6049,on,-4.243608
4267,he,-4.374016
898,an,-4.581176
4580,in,-4.769140


In [100]:
classe_2 = df[['Mots', '2']].sort_values(
    by='2',
    ascending=False
).head(20)

display(classe_2)

,Mots,2
3178,electrocuted,4.879844
6626,power,3.828032
3253,energized,3.409463
2305,contacted,2.919579
5233,line,2.737297
9518,volt,2.648479
3170,electrical,2.515820
6150,overhead,2.417353
7836,shock,2.207810
2304,contact,2.149902


La classe 2 correspond à des accidents impliquant de l'électricité. Électrocuté (electrocuted) est le mot possédant le coefficient le plus élevé de la classe.  

In [101]:
classe_3 = df[['Mots', '3']].sort_values(
    by='3',
    ascending=False
).head(20)

display(classe_3)

,Mots,3
3957,gas,2.359288
6140,overcome,2.092155
1051,asphyxia,2.044770
1744,carbon,2.000653
5728,monoxide,1.945791
4665,inhalation,1.755828
5417,manhole,1.598159
5589,methylene,1.511175
1953,chloride,1.511175
3443,exposure,1.505152


La classe 3 semble regrouper des évènements impliquant des gaz ayant pu causer une incommodation (overcome) ou une asphyxie.  Nous remarquons également la présence de noms de gaz dans  la liste comme carbon, monoxide, methylene, chloride, anhydrous ou ammoniac. 

In [102]:
classe_4 = df[['Mots', '4']].sort_values(
    by='4',
    ascending=False
).head(20)

display(classe_4)

,Mots,4
4288,heat,3.721182
4436,hot,2.879560
8771,temperature,2.584734
3515,fahrenheit,2.481226
8317,steam,2.445942
2701,degrees,2.262263
3403,exhaustion,1.980855
1625,burned,1.928799
6302,patcher,1.607903
2700,degree,1.607521


La classe 4 semble être associée à des accidents liés à de fortes températures. Les mots ayant le coefficient le plus élevés sont chaleur (heat) et chaud (hot). Nous remarquons également des mots comme asphalt et paint, qui pourraient correspondre à des contextes de travail impliquant une exposition à la chaleur. L'année 2005 figure aussi dans les coefficients les plus élevés. Il serait intéressant de vérifier si cette année est associés à des températures plus élevés qu'à la normale.

In [103]:
classe_5 = df[['Mots', '5']].sort_values(
    by='5',
    ascending=False
).head(20)

display(classe_5)

,Mots,5
3587,fell,4.304957
7413,roof,3.081931
3896,ft,2.875932
7569,scaffold,2.628302
3529,fall,2.580951
5068,ladder,2.540365
4267,he,2.171720
3586,feet,2.026626
3731,floor,1.698090
6049,on,1.670585


La classe 5 semble correspondre à des chutes. Cette classe possède le plus grand nombre d'exemple, soit 958. Certains mots ayant des coefficients élevés demeurent très généraux comme he, to et on. Cependant, le contexte demeure identifiable grâce à des mots comme est tombé (fell), chute (fall) et toit (roof).

In [104]:
classe_6 = df[['Mots', '6']].sort_values(
    by='6',
    ascending=False
).head(20)

display(classe_6)

,Mots,6
1630,burns,3.818248
3649,fire,3.648203
3957,gas,3.231116
3436,explosion,2.891836
4534,ignited,2.650497
8699,tank,2.139983
3433,exploded,1.960480
3898,fuel,1.929632
992,arc,1.704864
2700,degree,1.642530


La classe 6 semble représenter des accidents impliquant des brulures, du feu, des explosions ou des incendies. 

In [105]:
classe_7 = df[['Mots', '7']].sort_values(
    by='7',
    ascending=False
).head(20)

display(classe_7)

,Mots,7
7559,saw,2.354285
3638,finger,2.084927
4363,his,1.875472
4192,hand,1.759768
1382,blade,1.517356
5170,left,1.170093
1177,backhoe,0.997015
4149,gun,0.910297
5821,nail,0.887152
1990,circular,0.826461


La classe 7 semble représenter des accidents impliquant des coupures ou des amputations causées par des outils. Parmi les mots importants, nous retrouvons notamment « scie » (saw) ainsi que plusieurs parties du corps fréquemment touchées, comme les doigts (finger), les ongles (nail) et les mains (hand).

In [106]:
classe_8 = df[['Mots', '8']].sort_values(
    by='8',
    ascending=False
).head(20)

display(classe_8)

,Mots,8
8443,struck,3.195944
1278,beam,2.009337
2460,crane,1.833078
8423,striking,1.746683
8824,the,1.621200
4268,head,1.600588
5268,load,1.556628
3190,elevator,1.548847
1660,by,1.479728
9078,tree,1.409842


La classe 8 semble représenter des accidents de travail dans lesquels un travailleur a été frappé ou heurté (struck) par un objet comme un ascenceur(elevator), un arbre (tree), un tuyau (pipe) ou une grue (crane). Les mots les plus fréquents suggèrent aussi que les blessures sont souvent situées à la tête ou aux yeux. 

## Section 4 - Section réservée pour nos tests